# Count portion error analysis

Filterable dataframe for **count** ingredient lines from the 1,000-recipe feasibility run
(`portion_feasibility_1000/`), merged with parse fields, LLM fdc judge output, and USDA
count-portion availability for the matched food.

**Kernel cwd:** `Capstone/` or `scratch/EDA/`.

**Workflow:** run the build cell once, then filter `count_eda` in the cells below
(or set `FILTER_*` variables and re-run the display cell).

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def resolve_capstone_root() -> Path:
    cwd = Path.cwd()
    for candidate in (cwd, cwd.parent, cwd.parent.parent):
        if (candidate / "scripts" / "db.py").is_file():
            return candidate
    raise FileNotFoundError("Run with kernel cwd = Capstone/ or scratch/EDA/")


ROOT = resolve_capstone_root()
SCRIPTS = ROOT / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from amount_kind import infer_count_query
from db import connect, load_dotenv
from portion_gram import build_count_portion_index, classify_food_portion_row, _load_portion_rows_for_fdc

load_dotenv(ROOT / ".env")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

FEAS_DIR = ROOT / "scratch" / "EDA" / "portion_feasibility_1000"
CACHE_PATH = FEAS_DIR / "count_portion_eda.parquet"
print(f"Capstone root: {ROOT}")
print(f"Feasibility dir: {FEAS_DIR}")

Capstone root: /Users/danielcosta/Berkeley/Capstone
Feasibility dir: /Users/danielcosta/Berkeley/Capstone/scratch/EDA/portion_feasibility_1000


## Build `count_eda` dataframe

Set `REBUILD = True` to refresh from Postgres + judge artifacts. Default loads cached parquet.

In [2]:
REBUILD = False  # True to re-query USDA count portions

JUDGE_COLS = [
    "llm_fdc_id", "llm_description", "llm_certainty", "llm_rationale", "llm_error",
    "llm_abstained", "llm_agrees_with_staged", "llm_pick_has_count_portion",
    "staged_fdc_id", "staged_description", "retrieval_tier", "portion_filter_kind",
    "grams", "grams_status", "grams_method",
]

PARSE_COLS = [
    "quantity", "unit", "unit_raw", "name", "preparation", "parse_status",
    "amount_kind", "amount_kind_final", "amount_kind_source", "needs_portion",
]


def _count_query_token(unit, name, ingredient: str) -> str | None:
    tokens = infer_count_query(unit, name)
    if tokens:
        return tokens[0]
    return None


def _usda_count_portion_summary(conn, fdc_id: int | None, count_index: dict) -> tuple[int, str]:
    if fdc_id is None or pd.isna(fdc_id):
        return 0, ""
    fid = int(fdc_id)
    candidates = count_index.get(fid, [])
    if candidates:
        labels = []
        for c in candidates[:8]:
            label = c.count_label or c.modifier or c.portion_description or "?"
            labels.append(f"{label} ({c.gram_weight}g)")
        extra = len(candidates) - 8
        text = " | ".join(labels)
        if extra > 0:
            text += f" | +{extra} more"
        return len(candidates), text
    rows = _load_portion_rows_for_fdc(conn, fid)
    count_rows = [r for r in rows if classify_food_portion_row(r) == "count"]
    if not count_rows:
        return 0, ""
    labels = []
    for r in count_rows[:8]:
        mu = r.get("measure_unit_name") or ""
        mod = r.get("modifier") or ""
        desc = r.get("portion_description") or ""
        labels.append(f"{mu}/{mod}/{desc} ({r.get('gram_weight')}g)".strip("/"))
    return len(count_rows), " | ".join(labels)


def build_count_eda(*, rebuild: bool = False) -> pd.DataFrame:
    if not rebuild and CACHE_PATH.is_file():
        df = pd.read_parquet(CACHE_PATH)
        print(f"Loaded cached count_eda ({len(df):,} rows) → {CACHE_PATH}")
        return df

    amount = pd.read_parquet(FEAS_DIR / "amount_classification.parquet")
    judge = pd.read_parquet(FEAS_DIR / "judge_matches_raw.parquet")

    base = amount.merge(
        judge[["recipe_id", "ingredient_idx", "ingredient"] + JUDGE_COLS],
        on=["recipe_id", "ingredient_idx", "ingredient"],
        how="inner",
    )
    df = base[base["amount_kind_final"] == "count"].copy()

    df["count_query_token"] = [
        _count_query_token(r.unit, r.name, r.ingredient)
        for r in df.itertuples(index=False)
    ]
    df["has_fdc"] = df["llm_fdc_id"].notna()
    df["has_grams"] = df["grams"].notna()
    df["resolved_both"] = df["has_fdc"] & df["has_grams"]

    def _bucket(row) -> str:
        status = row.get("grams_status")
        if status == "ok_count_portion":
            return "ok_count_portion"
        if status in (
            "unresolvable_serving_only",
            "ambiguous_accepted",
            "vague_amount",
            "compound_skipped",
            "ok_embedded_mass",
        ):
            return str(status)
        if not row["has_fdc"]:
            return "missing_fdc"
        if status == "no_portion":
            if row.get("llm_pick_has_count_portion"):
                return "no_portion_fdc_has_count_rows"
            return "no_portion_fdc_lacks_count_rows"
        return str(status)

    df["error_bucket"] = [_bucket(r) for r in df.to_dict(orient="records")]

    with connect() as conn:
        count_index = build_count_portion_index(conn)
        n_rows: list[int] = []
        labels: list[str] = []
        for r in df.itertuples(index=False):
            n, text = _usda_count_portion_summary(conn, r.llm_fdc_id, count_index)
            n_rows.append(n)
            labels.append(text)

    df["n_usda_count_portions"] = n_rows
    df["usda_count_portion_labels"] = labels

    df.to_parquet(CACHE_PATH, index=False)
    print(f"Built count_eda ({len(df):,} rows) → {CACHE_PATH}")
    return df


count_eda = build_count_eda(rebuild=REBUILD)
DISPLAY_COLS = [
    "recipe_id", "ingredient_idx", "ingredient", "quantity", "unit", "name",
    "count_query_token", "error_bucket", "grams_status",
    "llm_fdc_id", "llm_description", "llm_pick_has_count_portion",
    "n_usda_count_portions", "usda_count_portion_labels",
    "grams", "llm_certainty", "llm_rationale", "llm_error",
    "staged_fdc_id", "llm_agrees_with_staged", "retrieval_tier",
]
DISPLAY_COLS = [c for c in DISPLAY_COLS if c in count_eda.columns]

Loaded cached count_eda (2,379 rows) → /Users/danielcosta/Berkeley/Capstone/scratch/EDA/portion_feasibility_1000/count_portion_eda.parquet


In [3]:
print(f"Count lines: {len(count_eda):,}")
print(f"Resolved (fdc + grams): {count_eda['resolved_both'].sum():,} ({100 * count_eda['resolved_both'].mean():.1f}%)")
print()
print("error_bucket:")
display(count_eda["error_bucket"].value_counts().to_frame("n"))
print("grams_status:")
display(count_eda["grams_status"].value_counts().to_frame("n"))

Count lines: 2,379
Resolved (fdc + grams): 442 (18.6%)

error_bucket:


,n
error_bucket,
no_portion_fdc_lacks_count_rows,1016
no_portion_fdc_has_count_rows,815
ok_count_portion,442
missing_fdc,106


grams_status:


,n
grams_status,
no_portion,1831
ok_count_portion,442
missing_fdc,106


## Filter & browse

Edit the variables below, then re-run the next cell.

In [25]:
# --- filter knobs (set to None to ignore) ---
FILTER_ERROR_BUCKET = "no_portion_fdc_has_count_rows"  # e.g. "no_portion_fdc_lacks_count_rows", "missing_fdc", "ok_count_portion"
FILTER_GRAMS_STATUS = None          # e.g. "no_portion"
FILTER_HAS_FDC = True            # True / False / None
FILTER_RESOLVED_BOTH = False        # True / False / None
FILTER_INGREDIENT_CONTAINS = None   # e.g. "egg", "clove", "slice"
FILTER_UNIT_CONTAINS = None         # e.g. "can", "package"
FILTER_MIN_USDA_COUNT_PORTIONS = None  # e.g. 1 to require USDA count rows on matched fdc
FILTER_LLM_HAS_COUNT_FLAG = None    # True = llm_pick_has_count_portion
FILTER_RECIPE_ID = None             # e.g. 1702
SORT_BY = ["error_bucket", "ingredient"]  # or ["n_usda_count_portions"]
HEAD_N = None                       # None to show all matching rows


def apply_filters(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if FILTER_ERROR_BUCKET is not None:
        out = out[out["error_bucket"] == FILTER_ERROR_BUCKET]
    if FILTER_GRAMS_STATUS is not None:
        out = out[out["grams_status"] == FILTER_GRAMS_STATUS]
    if FILTER_HAS_FDC is not None:
        out = out[out["has_fdc"] == FILTER_HAS_FDC]
    if FILTER_RESOLVED_BOTH is not None:
        out = out[out["resolved_both"] == FILTER_RESOLVED_BOTH]
    if FILTER_INGREDIENT_CONTAINS:
        out = out[out["ingredient"].str.contains(FILTER_INGREDIENT_CONTAINS, case=False, na=False)]
    if FILTER_UNIT_CONTAINS:
        out = out[out["unit"].astype(str).str.contains(FILTER_UNIT_CONTAINS, case=False, na=False)]
    if FILTER_MIN_USDA_COUNT_PORTIONS is not None:
        out = out[out["n_usda_count_portions"] >= FILTER_MIN_USDA_COUNT_PORTIONS]
    if FILTER_LLM_HAS_COUNT_FLAG is not None:
        out = out[out["llm_pick_has_count_portion"] == FILTER_LLM_HAS_COUNT_FLAG]
    if FILTER_RECIPE_ID is not None:
        out = out[out["recipe_id"] == FILTER_RECIPE_ID]
    if SORT_BY:
        out = out.sort_values(SORT_BY, na_position="last")
    return out


DISPLAY_COLS = [
    'ingredient', 'quantity', 'unit', 'name', 'error_bucket', 'grams_status',
    'llm_fdc_id', 'llm_description', 'n_usda_count_portions', 'usda_count_portion_labels', 'grams', 'llm_certainty'
]


filtered = apply_filters(count_eda)
print(f"Matching rows: {len(filtered):,} / {len(count_eda):,}")
view = filtered[DISPLAY_COLS] if HEAD_N is None else filtered[DISPLAY_COLS].head(HEAD_N)
display(view.head(50))

Matching rows: 815 / 2,379


,ingredient,quantity,unit,name,error_bucket,grams_status,llm_fdc_id,llm_description,n_usda_count_portions,usda_count_portion_labels,grams,llm_certainty
25,1 (1 lb.) can apricot halves,1.0,each,(1 lb.) can apricot halves,no_portion_fdc_has_count_rows,no_portion,171698.0,"Apricots, canned, water pack, with skin, solids and liquids",1,apricot half with liquid (36.0g),NaN,0.90
2350,1 (1 ounce) package pork gravy mix (Pork flavored-just add water mix),1.0,each,(1 ounce) package pork gravy mix (Pork flavored-just add water mix),no_portion_fdc_has_count_rows,no_portion,171173.0,"Gravy, pork, dry, powder",1,serving (6.7g),NaN,0.90
520,1 (1 oz.) pkg. Hidden Valley Ranch original salad dressing mix,1.0,each,(1 oz.) pkg. Hidden Valley Ranch original salad dressing mix,no_portion_fdc_has_count_rows,no_portion,173592.0,"Salad dressing, ranch dressing, regular",1,serving (30.0g),NaN,0.90
1519,1 (1-pound) flank steak,1.0,each,(1-pound) flank steak,no_portion_fdc_has_count_rows,no_portion,169433.0,"Beef, flank, steak, separable lean and fat, trimmed to 0"" fat, choice, raw",1,steak (202.0g),NaN,0.90
1858,"1 (10 ounce) jar maraschino cherries, drained and juice reserved",1.0,each,"(10 ounce) jar maraschino cherries, and juice reserved",no_portion_fdc_has_count_rows,no_portion,167766.0,"Maraschino cherries, canned, drained",1,cherry (nlea serving) (5.0g),NaN,0.90
285,1 (10 oz.) angel food cake,1.0,each,(10 oz.) angel food cake,no_portion_fdc_has_count_rows,no_portion,172694.0,"Cake, angelfood, commercially prepared",1,"cake (9"" dia x 4"") (340.0g)",NaN,0.90
14,1 (10 oz.) pkg. frozen whole kernel corn,1.0,each,(10 oz.) pkg. whole kernel corn,no_portion_fdc_has_count_rows,no_portion,169365.0,"Corn, sweet, white, frozen, kernels on cob, unprepared",1,"ear, yields (125.0g)",NaN,0.90
1889,1 (12 ounce) cancold beer,1.0,each,(12 ounce) cancold beer,no_portion_fdc_has_count_rows,no_portion,168746.0,"Alcoholic beverage, beer, regular, all",1,can (356.0g),NaN,0.90
647,1 (12 oz.) bite-size corn Chex,1.0,each,(12 oz.) bite-size corn Chex,no_portion_fdc_has_count_rows,no_portion,168127.0,"Snacks, corn-based, extruded, chips, unsalted",2,"bag, single serving (28.0g) | chips (18.0g)",NaN,0.80
352,1 (12 oz.) can corned beef,1.0,each,(12 oz.) can corned beef,no_portion_fdc_has_count_rows,no_portion,173860.0,"Corned beef loaf, jellied",1,slice (57.0g),NaN,0.85


In [27]:
terms = ['oz', 'ounce', 'lb.', 'pound', 'gram',]
pattern = '|'.join(terms)

filtered_view = view[
    (~view['ingredient'].str.contains(pattern, case=False, na=False)) &
    (view['ingredient'].str.contains('uff pastry'))
]

filtered_view

,ingredient,quantity,unit,name,error_bucket,grams_status,llm_fdc_id,llm_description,n_usda_count_portions,usda_count_portion_labels,grams,llm_certainty
1076,2 sheets puff pastry,2.0,each,sheets puff pastry,no_portion_fdc_has_count_rows,no_portion,172790.0,"Puff pastry, frozen, ready-to-bake",1,shell (47.0g),NaN,0.9
1660,2 sheets puff pastry,2.0,each,sheets puff pastry,no_portion_fdc_has_count_rows,no_portion,172790.0,"Puff pastry, frozen, ready-to-bake",1,shell (47.0g),NaN,0.9


## Quick views

Pre-built slices — run any cell to inspect a common failure mode.

In [ ]:
# Fdc matched, USDA has count portions, but rules still returned no_portion
display(
    count_eda[count_eda["error_bucket"] == "no_portion_fdc_has_count_rows"][DISPLAY_COLS].head(30)
)

In [ ]:
# Matched fdc has no count portion rows at all
display(
    count_eda[count_eda["error_bucket"] == "no_portion_fdc_lacks_count_rows"][DISPLAY_COLS].head(30)
)

In [ ]:
# No fdc match (judge abstain / error)
display(count_eda[~count_eda["has_fdc"]][DISPLAY_COLS].head(30))

In [ ]:
# Successfully resolved count → grams
display(count_eda[count_eda["resolved_both"]][DISPLAY_COLS].head(30))